### Étude d'impact : performance du contrôle de risque LTT en multi-classification

Ce notebook mesure comment le nombre de classes, la taille du jeu de calibration et le
niveau de risque cible affectent le contrôle de risque *Learn Then Test* (LTT) pour un
risque de précision one-vs-rest.

Pour chaque combinaison de paramètres, on génère un jeu de données synthétique, on
calibre un `MultiClassificationLTTController`, puis on compte combien de seuils de
probabilité (parmi ceux proposés dans `list_predict_params`) sont validés, c'est-à-dire
pour lesquels le risque est statistiquement contrôlé au niveau cible. Deux méthodes de
contrôle du FWER (*Family-Wise Error Rate*) sont comparées : `bonferroni_holm` et
`split_fixed_sequence`.

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from mapie.risk_control import MultiClassificationLTTController
from mapie.risk_control.risks import build_precision_ovr,build_abstention_rate_ovr,build_recall_ovr
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_colwidth", None)

#### Fonction d'expérience

`run_exp_multiclasse` génère un jeu de données synthétique à `n_classes` classes
(`data_per_classes` échantillons par classe), entraîne une régression logistique, puis
calibre un `MultiClassificationLTTController` sur le risque de précision one-vs-rest
(`build_precision_ovr`) pour `num_params` seuils de probabilité répartis entre 0.05 et
0.95. La fonction renvoie le nombre de seuils validés par LTT au niveau `target_level`.

Avec `fwer_method="split_fixed_sequence"`, l'ordre des paramètres doit être appris
(`learn_fixed_sequence_order`) sur un jeu de données indépendant du jeu de calibration
avant d'appeler `calibrate` : la fonction scinde alors `X_toy`/`y_toy` en deux pour cela.

In [4]:
n_lambdas = 5
lambda_1_values = np.linspace(0.3, 0.7, n_lambdas)
lambda_2_values = np.linspace(0.4, 0.8, n_lambdas)
to_explore = []
for lambda_1 in lambda_1_values:
    for lambda_2 in lambda_2_values:
        if lambda_2 >= lambda_1:
            to_explore.append((lambda_1, lambda_2))
to_explore = np.array(to_explore)

In [5]:
to_explore

array([[0.3, 0.4],
       [0.3, 0.5],
       [0.3, 0.6],
       [0.3, 0.7],
       [0.3, 0.8],
       [0.4, 0.4],
       [0.4, 0.5],
       [0.4, 0.6],
       [0.4, 0.7],
       [0.4, 0.8],
       [0.5, 0.5],
       [0.5, 0.6],
       [0.5, 0.7],
       [0.5, 0.8],
       [0.6, 0.6],
       [0.6, 0.7],
       [0.6, 0.8],
       [0.7, 0.7],
       [0.7, 0.8]])

In [6]:
n_classes = 5
target_level_score = [0.6]*n_classes
target_level_abstention = [0.5]*n_classes
list_target_level =  target_level_score +target_level_abstention + target_level_score
fwer_method = 'bonferroni_holm'
X_toy, y_toy = make_classification(
    n_samples=20000,
    n_features=10,
    n_informative=10,
    n_redundant=0,
    n_repeated=0,
    n_classes=n_classes,
    n_clusters_per_class=1,
    flip_y=0,
    random_state=42,
)
clf = LogisticRegression().fit(X_toy, y_toy)

# proba => seuil sur le softmax. 
def abstain_to_answer_multiclass(X, lambda_1, lambda_2):
    """Predict function with abstention based on two thresholds, for multiclass.
    - if the max probability (confidence in the most likely class) is below
      `lambda_1`, predict the most likely class
    - if the max probability is above `lambda_2`, predict the most likely class
    - if the max probability is between `lambda_1` and `lambda_2`, abstain (np.nan)
    """

    predictions_proba = clf.predict_proba(X)
    max_proba = predictions_proba.max(axis=1)
    class_label = predictions_proba.argmax(axis=1)
    y_pred = np.full_like(max_proba, np.nan)
    y_pred = np.where(max_proba <= lambda_1, np.nan , y_pred) # 0 is to be changed 
    y_pred = np.where(max_proba >= lambda_2, class_label, y_pred)
    return y_pred

# For split_fixed_sequence, the fixed-sequence ordering is learned starting
# from the first entry of list_predict_params, so we scan from high to low
# thresholds instead of low to high.

list_predict_params = to_explore
risks_precision = build_precision_ovr(np.unique(y_toy)) 
risks_abstention = build_abstention_rate_ovr(np.unique(y_toy))
risks_recall = build_recall_ovr(np.unique(y_toy))
risks = risks_precision  + risks_abstention +  risks_recall
#risks = build_abstention_rate_ovr(np.unique(y_toy))

In [7]:
mapie_clf = MultiClassificationLTTController(
    predict_function=abstain_to_answer_multiclass,
    target_level=list_target_level,
    risks=risks,
    confidence_level=0.8,
    best_predict_param_choice=risks_abstention[0],
    list_predict_params=list_predict_params,
    fwer_method=fwer_method,
)

# When fwer_method="split_fixed_sequence", the parameter order must be learned
# on data independent from the calibration set (see learn_fixed_sequence_order
# docstring).
mapie = mapie_clf.calibrate(X_toy, y_toy)

In [8]:
y_abstained = mapie_clf.predict(X_toy)

In [9]:
counts = pd.DataFrame(y_abstained).value_counts(dropna=False)
proportions = pd.DataFrame(y_abstained).value_counts(dropna=False, normalize=True)
pd.concat([counts, proportions], axis=1, keys=["count", "proportion"])

,count,proportion
0,,
0.0,4130,0.20650
2.0,4116,0.20580
1.0,3953,0.19765
3.0,3885,0.19425
4.0,3546,0.17730
NaN,370,0.01850


In [10]:
plot_df = pd.concat([counts, proportions], axis=1, keys=["count", "proportion"]).reset_index()
plot_df = plot_df.rename(columns={0: "classe"})
plot_df["type"] = np.where(plot_df["classe"].isna(), "abstention", "classe prédite")
plot_df["classe_label"] = plot_df["classe"].apply(
    lambda c: "abstention" if pd.isna(c) else str(c)
)
plot_df = plot_df.sort_values("classe", na_position="last")

fig = px.bar(
    plot_df,
    x="classe_label",
    y="proportion",
    color="type",
    color_discrete_map={"classe prédite": "green", "abstention": "red"},
    labels={"classe_label": "classe", "proportion": "proportion"},
    title="Proportion des classes prédites vs abstention",
)
fig.show()